In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
#import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import h5py
import copy

inFile = '../image_data/11162021_Bahareh-3155_EDTA.qptiff'
inFile_num = 3155
fabian_segmentation_file = '../segmentation_files/11162021_Bahareh-3155_EDTA_IBA1_cell1.h5'


xstart = 100
xstop = 1200
ystart = 100
ystop = 1200

In [ ]:
#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors

# now applying special thresholding

In [ ]:

sample_stats = {}
sample_stats['inFile'] = inFile
sample_stats['inFile_num'] = inFile_num
(all_pages,shapes)=tiff_in(inFile)
markerDict = channelNames(inFile)
sample_stats['proteins'] = list(markerDict.values())
print(sample_stats['proteins'])

In [ ]:
LimitsPaulaorig = {'DAPI_1':18, 'Neurofilament':0, 'Vimentin':0, 'Claudin-5':60, 'SMA':0, 'a-Beta':0, 'GFAP':0, 'Olig2':26, 'Synaptophysin':0,'ApoE':90, 'NeuN':0, 'H2A.x':0, 'CollagenIV':0, 'PCNA':175, 'MAP2':0}

LimitsPaulacustom = {'DAPI_1':25, 'Neurofilament':20, 'Vimentin':10, 'Claudin-5':60, 'SMA':20, 'a-Beta':18, 'GFAP':33, 
                     'Olig2':26, 'Synaptophysin':20,'ApoE':90, 'NeuN':30, 'H2A.x':15, 'CollagenIV':20, 'PCNA':175, 'MAP2':30}
# claudin 60
limit_choice = LimitsPaulacustom

for gene_name in limit_choice.keys():
    if gene_name not in sample_stats['proteins']:
        print(gene_name + ' not found')


sample_stats['proteins_used_in_clustering'] = list(limit_choice.keys())
sample_stats['detection_mins_used_in_clustering'] = list(limit_choice.values())
convolve_data = False
downscale_reduction = 1
sample_stats['reduction'] = downscale_reduction
mD = limit_choice
gaussian_filter = False
d = {}
shapes = []
dbinar = {}
for keyID in markerDict.keys():
    print(keyID)
    key = markerDict[keyID]
    if key in mD.keys():
         
        orig_Img = all_pages[0][keyID]
        print(orig_Img.shape)
        plot1 = plt.hist(orig_Img.flatten(), bins=100)
        tMin = mD[key]
        
        if (gaussian_filter):
            print('applying gaussian filter')
            orig_Img = gaussian(orig_Img, sigma=2, preserve_range=True)
        
            
        if convolve_data:
            print('reducing_dims')
            orig_Img = transform.pyramid_reduce(orig_Img, downscale = downscale_reduction, preserve_range=True)
            
        
        flat = orig_Img.flatten()
        flat = flat[flat >= tMin]
        
        a_max = min(np.percentile(flat, q = 99), tMin + 30)
        plt.axvline(x=tMin, color = 'black')
        plt.axvline(x=a_max, color = 'black')
        plt.yscale('log')
        plt.show()
        
        plot2 = imshow(orig_Img == 0, origin = 'lower')
        plt.show()
        
        
        inImg = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg[inImg < tMin] = 0

        shapes.append(inImg.shape)
        
        
        inImg = (inImg - inImg.min())/(inImg.max() - inImg.min())
        
        inImg = inImg.round(decimals = 3)
        d[key] = inImg.flatten()
        plot3 = imshow(inImg == 0, origin = 'lower')
        plt.show()
        print("Done clipping and thresholding",markerDict[keyID])
        
        
        inImg_binar = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg_binar[inImg_binar < tMin] = 0
        inImg_binar[inImg_binar > 0] = 1
        dbinar[key] = inImg_binar.flatten()
        
        
        
        
        
        
        
        
        
        
             
shape = shapes[0]
sample_stats['shape'] = shape
d['X'] = np.indices(shape)[0].flatten()
d['Y'] = np.indices(shape)[1].flatten()

df_sub = pd.DataFrame(data=d)


dbinar['X'] = np.indices(shape)[0].flatten()
dbinar['Y'] = np.indices(shape)[1].flatten()

df_sub_binar = pd.DataFrame(data = dbinar)

In [ ]:
for name in limit_choice.keys():
    print(name)
    imgdata = np.asarray(df_sub[name]).reshape(sample_stats['shape'])
    gooby =plt.imshow(imgdata[xstart:xstop,ystart:ystop], origin = 'lower')
    plt.savefig('paper_images/flowchart_protein_' + name + '.tiff', format='tiff')
    plt.savefig('paper_images/flowchart_protein_' + name + '.svg', format='svg')
    plt.show()
    gooby = plt.imshow(imgdata, origin='lower')
    
    plt.show()
    
    


In [ ]:
df_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_df.csv'

df_sub.to_csv(df_file_path)  

dfbinar_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_dfbinar.csv'

df_sub_binar.to_csv(dfbinar_file_path)

savepath = 'saved_clustering_images_and_tables/'
with open(savepath + str(sample_stats['inFile_num']) + '_sample_stats.txt', 'w') as dict_file:
            dict_file.write(json.dumps(sample_stats))


# load data

In [ ]:

savepath = 'saved_clustering_images_and_tables/'
with open(savepath + str(inFile_num) + '_sample_stats.txt') as f:
    data = f.read()
sample_stats = json.loads(data)
        
df_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_df.csv'
df_sub = pd.read_csv(df_file_path, index_col=0)

dfbinar_file_path = 'pre_clustering_dfs/' + str(sample_stats['inFile_num']) + '_pre_clustering_dfbinar.csv'
dfbinar_sub = pd.read_csv(dfbinar_file_path, index_col=0)



# add custom channels

In [ ]:

dfbinar_sub['SMA - Claudin5 +'] = (dfbinar_sub['SMA'] == 0) & (dfbinar_sub['Claudin-5'] == 1)
dfbinar_sub['GFAP + Vimentin +'] = (dfbinar_sub['GFAP'] == 1) & (dfbinar_sub['Vimentin'] == 1)


# add otsm annotations

In [ ]:
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict

In [ ]:
# read in fabian MG segmentations
with h5py.File(fabian_segmentation_file, 'r') as file:
    loaded_matrix = np.array(file['matrix'])
    
loaded_matrix = loaded_matrix.T
print(plt.imshow(loaded_matrix, origin = 'lower'))

dfbinar_sub['otsm_masks'] = loaded_matrix.flatten()

# read in segmentation metadata

fabian_data = anndata.read_h5ad('../fabian_metadata.h5ad')
fabian_metadata = fabian_data.obs

category_to_color = {'Grey Matter': 'red', 'White Matter': 'blue', np.NaN: 'Gray'}



# build new column in df sub for mapping

dfbinar_sub['ImageID_CellID_man'] = str(sample_stats['inFile_num']) + '_' + (dfbinar_sub['otsm_masks'].astype(int)).astype(str)
meta_sub = fabian_metadata[['ImageID_CellID', 'Parent']].set_index('ImageID_CellID')


keys = fabian_metadata['ImageID_CellID']
values = meta_sub['Parent']
dictionary = dict(zip(keys, values))
dfbinar_sub['Parent_from_metadata'] = [dictionary.get(val, -1) for val in dfbinar_sub['ImageID_CellID_man'].tolist()]

viz_col = copy.deepcopy(dfbinar_sub.otsm_masks)
viz_col[viz_col > 0] = 1
dfbinar_sub['viz_col'] = viz_col

# collect centroid locations

In [ ]:

meta_only_img = fabian_metadata[fabian_metadata['ImageIDOLD'] == str(sample_stats['inFile_num'])]
centroid_x = meta_only_img['spatial_X']/sample_stats['reduction']
centroid_y = meta_only_img['spatial_Y']/sample_stats['reduction']

color_vector = [category_to_color[category] for category in meta_only_img.Parent]
print(plt.scatter(x = meta_only_img.spatial_X, y = meta_only_img.spatial_Y, c = color_vector, s = 0.5))


imageid_cellid = meta_only_img['ImageID_CellID']
parent = meta_only_img['Parent']

In [ ]:
imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata, origin = 'lower')
plt.show()

In [ ]:
imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])

d = {'centroid_x': centroid_x, 'centroid_y': centroid_y}

neighborhood_table = pd.DataFrame(data=d)
#neighborhood_table = neighborhood_table[neighborhood_table['centroid_x'] < 1200]
neighborhood_table = neighborhood_table[(neighborhood_table['centroid_x'] < xstop) & (neighborhood_table['centroid_x'] > xstart)]

neighborhood_table = neighborhood_table[(neighborhood_table['centroid_y'] < ystop) & (neighborhood_table['centroid_y'] > ystart)]
plt.show()
plt.scatter(neighborhood_table['centroid_x'], neighborhood_table['centroid_y'])
plt.show()
implot = imshow(imgdata[xstart:xstop,ystart:ystop], origin = 'lower')
plt.scatter(neighborhood_table['centroid_x'] - xstart, neighborhood_table['centroid_y'] - xstart)
plt.show()


imgdata = np.asarray(dfbinar_sub['viz_col']).reshape([sample_stats['shape'][0], sample_stats['shape'][1]])
implot = imshow(imgdata, origin = 'lower')
plt.show()

# create cKDTrees from otsm points

In [ ]:
dfbinar_xy = dfbinar_sub[['Y', 'X']].to_numpy()

point_tree = scispatial.cKDTree(dfbinar_xy)
distance = 100
sample_stats['distance_used_in_neighborhood_analysis'] = distance

centers = np.column_stack((centroid_x, centroid_y))
neighbors = [point_tree.query_ball_point(centroid, distance) for centroid in centers]

d = {'imageid_cellid': imageid_cellid, 'parent':parent, 'centroid_x': centroid_x, 'centroid_y': centroid_y, 'neighbors': neighbors}
neighborhood_table = pd.DataFrame(data=d)

neighborhood_table['neighbor_pixel_ids'] = [dfbinar_sub.iloc[n].ImageID_CellID_man.tolist() for n in neighborhood_table.neighbors]


non_microglia_neighbors = []
non_microglia_neighbors_longboi = []

overlap_neighbors = []
overlap_neighbors_longboi = []

for ind in neighborhood_table.index:
    row = neighborhood_table.loc[ind]
    neighbors = row.neighbors
    neighbor_pixel_ids = row.neighbor_pixel_ids
    centroid_pixel_id = row.imageid_cellid
    neighbor_pixel_bools = [i != centroid_pixel_id for i in neighbor_pixel_ids]
    new_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if v]
    overlap_list = [i for (i,v) in zip (neighbors, neighbor_pixel_bools) if not v]
    non_microglia_neighbors.append(new_list)
    non_microglia_neighbors_longboi += new_list
    
    overlap_neighbors.append(overlap_list)
    overlap_neighbors_longboi += overlap_list

neighborhood_table['non_microglia_neighbors'] = non_microglia_neighbors
neighborhood_table['overlap_neighbors'] = overlap_neighbors





In [ ]:

dfbinar_sub['is_neighborhood'] = 0
dfbinar_sub.is_neighborhood.iloc[non_microglia_neighbors_longboi] = 1
imgdata = np.asarray(dfbinar_sub['is_neighborhood']).reshape(sample_stats['shape'])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])
plt.savefig('paper_images/flowchart_radius.tiff', format='tiff')
plt.savefig('paper_images/flowchart_radius.svg', format='svg')

dfbinar_sub['is_neighborhood'] = 0
dfbinar_sub.is_neighborhood.iloc[overlap_neighbors_longboi] = 1
imgdata = np.asarray(dfbinar_sub['is_neighborhood']).reshape(sample_stats['shape'])
implot = imshow(imgdata[xstart:xstop,ystart:ystop])
plt.savefig('paper_images/flowchart_segment.tiff', format='tiff')
plt.savefig('paper_images/flowchart_segment.svg', format='svg')



# VIZ

In [ ]:


# Create subplots to overlay the images

custom_boi = list(set(non_microglia_neighbors_longboi) - set(overlap_neighbors_longboi))
dfbinar_sub['is_neighborhood'] = 0
dfbinar_sub.is_neighborhood.iloc[custom_boi] = 1
image1 = np.asarray(dfbinar_sub['is_neighborhood']).reshape(sample_stats['shape']) * 1


# make flowchart image
image2 = np.asarray(dfbinar_sub['otsm_masks']).reshape(sample_stats['shape'])
image2 = (image2 > 0).astype(int) * 7

# Plot the second image using imshow, adjusting alpha for transparency


image3 = np.asarray(dfbinar_sub['a-Beta']).reshape(sample_stats['shape']) * 3

image4 = np.asarray(dfbinar_sub['GFAP']).reshape(sample_stats['shape']) * 4

image5 = np.asarray(dfbinar_sub['CollagenIV']).reshape(sample_stats['shape']) * 2

image6 = np.asarray(dfbinar_sub['Neurofilament']).reshape(sample_stats['shape']) * 5

image7 = np.asarray(dfbinar_sub['NeuN']).reshape(sample_stats['shape']) * 6


# Overlay the second image with transparency (adjust alpha as needed)




import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np



# Create a figure and set the axis to be transparent
fig, ax = plt.subplots()

# Layer the images with colors
layered_image = np.maximum(image1, image2)
layered_image = np.maximum(layered_image, image3)
layered_image = np.maximum(layered_image, image4)
layered_image = np.maximum(layered_image, image5)
layered_image = np.maximum(layered_image, image6)
layered_image = np.maximum(layered_image, image7)


from matplotlib.colors import LinearSegmentedColormap
# Create a custom colormap with distinct colors for each value
cMapDict = {0: 'white',
 1: 'blue',
 2: '#279e68',
 3: '#d62728',
 4: '#aa40fc',
 5: 'orange',
 6: 'gray',
 7:'yellow'
 }



colors_list = [cMapDict[i] for i in range(len(cMapDict))]
custom_cmap = ListedColormap(colors_list)

# Display the layered image
ax.imshow(layered_image[xstart:xstop,ystart:ystop], cmap=custom_cmap, interpolation='nearest')
ax.axis('off')  # Turn off the axis
plt.savefig('paper_images/flowchart_image.tiff', format='tiff')
plt.savefig('paper_images/flowchart_image.svg', format='svg')

plt.show()



